# ExamGuard: train the fusion models

This notebook trains the pair and incident priority models from the exported synthetic cohorts.

**What the score means.** The output is a 0-100 review priority that routes work to a human
reviewer. It is not a probability that a candidate committed misconduct.

**What the data means.** Labels come from the ExamGuard simulator. Metrics measured here show
that the pipeline is wired correctly. They are not evidence of real-world validity. Re-validate
on a consented pilot before enabling `USE_LEARNED_FUSION` in production.

In [ ]:
import json
import shutil
import subprocess
import sys
import tarfile
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")
print("input entries:", sorted(p.name for p in INPUT_ROOT.glob("*")))
manifest_paths = sorted(INPUT_ROOT.rglob("manifest.json"))
assert manifest_paths, "attach the ExamGuard training-rows dataset to this notebook"
DATASET_DIR = manifest_paths[0].parent
print("dataset:", DATASET_DIR)

WORK = Path("/kaggle/working/ai")
if WORK.exists():
    shutil.rmtree(WORK)
(WORK / "data").mkdir(parents=True)
if (DATASET_DIR / "code.tar.gz").exists():
    with tarfile.open(DATASET_DIR / "code.tar.gz") as bundle:
        bundle.extractall(WORK, filter="data")
else:
    shutil.copytree(DATASET_DIR / "code" / "app", WORK / "app")
for name in ("pairs.jsonl", "incidents.jsonl", "manifest.json"):
    shutil.copy2(DATASET_DIR / name, WORK / "data" / name)

manifest = json.loads((WORK / "data" / "manifest.json").read_text())
print(json.dumps({k: manifest[k] for k in ("generatorVersion", "seed", "synthetic")}, indent=2))
for kind in ("pair", "incident"):
    part = manifest[kind]
    print(f"{kind}: {part['rows']} rows, {part['positives']} positives, {part['positiveRate']:.1%}")


In [ ]:
SEED = manifest["seed"]
MODELS = Path("/kaggle/working/models")
MODELS.mkdir(exist_ok=True)

result = subprocess.run(
    [sys.executable, "-m", "app.training.train", "--data", str(WORK / "data"), "--out", str(MODELS), "--seed", str(SEED)],
    cwd=WORK,
    text=True,
    capture_output=True,
)
print(result.stdout)
print(result.stderr)
result.check_returncode()

In [ ]:
for kind in ("pair", "incident"):
    meta = json.loads((MODELS / f"{kind}_fusion.meta.json").read_text())
    metrics = meta["metrics"]
    print(f"== {kind} ==")
    print("  average precision :", metrics["averagePrecision"])
    print("  ROC AUC          :", metrics["rocAuc"])
    print("  precision @ 10%  :", metrics["precisionAtBudget"])
    print("  threshold        :", meta["threshold"])
    print("  test rows        :", metrics["rows"])

print(sorted(path.name for path in MODELS.iterdir()))

## Using the artifacts

Download `pair_fusion.joblib`, `incident_fusion.joblib` and `MODEL_CARD.md` from the notebook
output, place them in `ai/models/`, and set `USE_LEARNED_FUSION=true`. Keep the model card with
the artifacts so the synthetic-only provenance travels with the numbers.

To version them on Kaggle instead, run locally:

```bash
python -m app.training.kaggle_publish models --models ai/models
```